# Next-Day Stock Volatility Forecasting (LSTM)

This notebook builds one-step-ahead forecasts for next-day `volatility_1D` for:
- `CTVA`
- `DDOG`

We train one separate LSTM model per stock, compare against a naive baseline, retrain once on train+validation, and then forecast the unseen period sequentially without daily refits.

## Workflow Overview

1. Load and sort data
2. Apply the exact date split logic provided
3. Filter to `CTVA` and `DDOG`
4. Build leakage-safe engineered features
5. Build sequence windows for LSTM
6. Train and validate:
   - Naive baseline (`today volatility -> next day volatility`)
   - LSTM model
7. Retrain LSTM on train+validation
8. Forecast unseen one day ahead sequentially, without refitting
9. Report metrics and plots

## Imports and Reproducibility

In [ ]:
import os
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

try:
    import tensorflow as tf
    from tensorflow.keras import Sequential
    from tensorflow.keras.layers import Input, LSTM, Dense
    from tensorflow.keras.callbacks import EarlyStopping
except ModuleNotFoundError as e:
    raise ModuleNotFoundError(
        "TensorFlow is required for this notebook. Install it in your Jupyter kernel environment, for example: `%pip install tensorflow`."
    ) from e

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

print(f"Seed set to: {SEED}")

## Paths and Global Configuration

In [ ]:
csv_path = "data/sp500_5yr_with_sectors_weights.csv"
unseen_path = "data/sp500_unseen.csv"

symbols_to_model = ["CTVA", "DDOG"]
include_today = True
use_log_volatility = False

lookback = 21

lstm_units = 32
epochs = 100
batch_size = 32
patience = 10
learning_rate = 1e-3

print("csv_path:", csv_path)
print("unseen_path:", unseen_path)
print("symbols:", symbols_to_model)
print("include_today:", include_today)
print("use_log_volatility:", use_log_volatility)
print("lookback:", lookback)

## Load Data and Inspect Raw Structure

In [ ]:
df_initial = pd.read_csv(csv_path)
df_initial["Date"] = pd.to_datetime(df_initial["Date"])
df_initial = df_initial.sort_values(["Symbol", "Date"]).copy()

unseen_reference = pd.read_csv(unseen_path)
unseen_reference["Date"] = pd.to_datetime(unseen_reference["Date"])
unseen_reference = unseen_reference.sort_values(["Symbol", "Date"]).copy()

print("Main data shape:", df_initial.shape)
print("Unseen reference shape:", unseen_reference.shape)
print("\nMain columns:")
print(df_initial.columns.tolist())

In [ ]:
print("Main date range:", df_initial["Date"].min().date(), "to", df_initial["Date"].max().date())
print("Unseen reference date range:", unseen_reference["Date"].min().date(), "to", unseen_reference["Date"].max().date())
print("\nHead of main data:")
display(df_initial.head(3))

## Exact Date Split Logic (As Requested)

The following cell uses the split code exactly as provided.

In [ ]:
df = pd.read_csv(csv_path)
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values(['Symbol', 'Date']).copy()

unseen_end = df['Date'].max().normalize()
unseen_start = unseen_end - pd.DateOffset(years=1) + pd.Timedelta(days=1)
val_end = unseen_start - pd.Timedelta(days=1)
val_start = val_end - pd.DateOffset(years=1) + pd.Timedelta(days=1)
train_end = val_start - pd.Timedelta(days=1)
data_start = train_end - pd.DateOffset(years=3) + pd.Timedelta(days=1)

print("data_start   :", data_start.date())
print("train_end    :", train_end.date())
print("val_start    :", val_start.date())
print("val_end      :", val_end.date())
print("unseen_start :", unseen_start.date())
print("unseen_end   :", unseen_end.date())

## Filter to CTVA and DDOG

In [ ]:
df = df[df["Symbol"].isin(symbols_to_model)].copy()
df = df.sort_values(["Symbol", "Date"]).reset_index(drop=True)

unseen_reference_small = unseen_reference[unseen_reference["Symbol"].isin(symbols_to_model)].copy()
unseen_reference_small = unseen_reference_small.sort_values(["Symbol", "Date"]).reset_index(drop=True)

print("Filtered main shape:", df.shape)
print("Filtered unseen shape:", unseen_reference_small.shape)

display(
    df.groupby("Symbol")
      .agg(rows=("Date", "size"), min_date=("Date", "min"), max_date=("Date", "max"))
      .reset_index()
)

## Leakage-Safe Stage Separation

We isolate unseen first and keep it out of model development.

In [ ]:
pre_unseen_raw = df[(df["Date"] >= data_start) & (df["Date"] <= val_end)].copy()
unseen_raw_from_split = df[(df["Date"] >= unseen_start) & (df["Date"] <= unseen_end)].copy()
unseen_raw = unseen_reference_small[
    (unseen_reference_small["Date"] >= unseen_start) & (unseen_reference_small["Date"] <= unseen_end)
].copy()

print("pre_unseen_raw shape:", pre_unseen_raw.shape)
print("unseen_raw_from_split shape:", unseen_raw_from_split.shape)
print("unseen_raw (from unseen_path) shape:", unseen_raw.shape)

split_counts = unseen_raw_from_split.groupby("Symbol").size().rename("rows_split")
file_counts = unseen_raw.groupby("Symbol").size().rename("rows_file")
print("\nUnseen row-count comparison by symbol:")
display(pd.concat([split_counts, file_counts], axis=1).fillna(0).astype(int).reset_index())

## Feature Engineering Functions

In [ ]:
def create_engineered_features(
    input_df: pd.DataFrame,
    include_today: bool = True,
    use_log_volatility: bool = False
) -> pd.DataFrame:
    out_frames = []

    for symbol, g in input_df.sort_values(["Symbol", "Date"]).groupby("Symbol", sort=False):
        g = g.copy()
        g["Close"] = g["Close"].astype(float)
        g["Volume"] = g["Volume"].astype(float)

        # Daily return and volatility proxy
        g["log_return_1D"] = np.log(g["Close"]).diff()
        g["volatility_1D"] = g["log_return_1D"].abs()
        if use_log_volatility:
            g["volatility_1D"] = np.log1p(g["volatility_1D"])

        # Daily log volume change
        log_volume = np.log(g["Volume"].replace(0, np.nan))
        g["volume_change_1D"] = log_volume.diff()

        # Rolling features
        g["log_return_5D"] = g["log_return_1D"].rolling(5, min_periods=5).sum()
        g["log_return_21D"] = g["log_return_1D"].rolling(21, min_periods=21).sum()

        g["volatility_5D"] = g["log_return_1D"].rolling(5, min_periods=5).std()
        g["volatility_21D"] = g["log_return_1D"].rolling(21, min_periods=21).std()

        g["volume_change_5D"] = g["volume_change_1D"].rolling(5, min_periods=5).sum()
        g["volume_change_21D"] = g["volume_change_1D"].rolling(21, min_periods=21).sum()

        # Forecast target: next-day volatility
        g["target_next_volatility_1D"] = g["volatility_1D"].shift(-1)

        out_frames.append(g)

    engineered = pd.concat(out_frames, axis=0).sort_values(["Symbol", "Date"]).reset_index(drop=True)
    return engineered


def make_lstm_sequences(
    input_df: pd.DataFrame,
    feature_cols,
    target_col: str,
    lookback: int
):
    g = input_df.sort_values("Date").copy().reset_index(drop=True)

    X_list, y_list, meta_list = [], [], []
    feat = g[feature_cols].values
    targ = g[target_col].values

    for end_idx in range(lookback - 1, len(g)):
        start_idx = end_idx - lookback + 1
        x_win = feat[start_idx:end_idx + 1]
        y_val = targ[end_idx]

        if np.isnan(x_win).any() or np.isnan(y_val):
            continue

        X_list.append(x_win)
        y_list.append(y_val)
        meta_list.append({
            "Date": g.loc[end_idx, "Date"],
            "Symbol": g.loc[end_idx, "Symbol"],
            "volatility_1D": g.loc[end_idx, "volatility_1D"],
        })

    if len(X_list) == 0:
        X = np.empty((0, lookback, len(feature_cols)), dtype=float)
        y = np.empty((0,), dtype=float)
        meta = pd.DataFrame(columns=["Date", "Symbol", "volatility_1D"])
    else:
        X = np.asarray(X_list, dtype=float)
        y = np.asarray(y_list, dtype=float)
        meta = pd.DataFrame(meta_list)

    return X, y, meta

In [ ]:
feature_cols = [
    "log_return_1D",
    "volatility_1D",
    "volume_change_1D",
    "log_return_5D",
    "log_return_21D",
    "volatility_5D",
    "volatility_21D",
    "volume_change_5D",
    "volume_change_21D",
]
target_col = "target_next_volatility_1D"

print("Number of per-timestep features:", len(feature_cols))
print("Features:", feature_cols)

## Build Pre-Unseen Feature Block Then Split Train/Validation

Per leakage rules:
- engineer features on full pre-unseen block first
- then split engineered rows by date into train and validation

In [ ]:
pre_unseen_features = create_engineered_features(
    pre_unseen_raw,
    include_today=include_today,
    use_log_volatility=use_log_volatility,
)

model_ready_pre_unseen = pre_unseen_features.dropna(subset=feature_cols + [target_col]).copy()

train_df = model_ready_pre_unseen[
    (model_ready_pre_unseen["Date"] >= data_start) & (model_ready_pre_unseen["Date"] <= train_end)
].copy()
val_df = model_ready_pre_unseen[
    (model_ready_pre_unseen["Date"] >= val_start) & (model_ready_pre_unseen["Date"] <= val_end)
].copy()

print("pre_unseen_features shape:", pre_unseen_features.shape)
print("model_ready_pre_unseen shape:", model_ready_pre_unseen.shape)
print("train_df shape:", train_df.shape)
print("val_df shape  :", val_df.shape)

display(train_df.head(3))

In [ ]:
print("Train rows by stock:")
display(
    train_df.groupby("Symbol")
    .agg(rows=("Date", "size"), min_date=("Date", "min"), max_date=("Date", "max"))
    .reset_index()
)

print("Validation rows by stock:")
display(
    val_df.groupby("Symbol")
    .agg(rows=("Date", "size"), min_date=("Date", "min"), max_date=("Date", "max"))
    .reset_index()
)

## Metrics and Model Helper Functions

In [ ]:
def smape(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    denom = np.abs(y_true) + np.abs(y_pred)
    denom = np.where(denom == 0, 1e-8, denom)
    return 100 * np.mean(2 * np.abs(y_pred - y_true) / denom)


def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = float(np.sqrt(mse))
    mae = mean_absolute_error(y_true, y_pred)

    if len(y_true) > 1 and np.std(y_true) > 0 and np.std(y_pred) > 0:
        corr = float(np.corrcoef(y_true, y_pred)[0, 1])
    else:
        corr = np.nan

    return {
        "RMSE": rmse,
        "MAE": float(mae),
        "MSE": float(mse),
        "Corr": corr,
        "sMAPE": float(smape(y_true, y_pred)),
    }


def build_lstm_model(lookback: int, n_features: int, lstm_units: int = 32, learning_rate: float = 1e-3):
    model = Sequential([
        Input(shape=(lookback, n_features)),
        LSTM(lstm_units),
        Dense(1, activation="softplus")
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss="mse"
    )
    return model

## Validation Stage: Baseline and LSTM (Per Stock)

Scaling rule:
- Fit `StandardScaler` on train only (per stock), then transform train and validation.

Sequence rule:
- Validation sequences include trailing train history (`lookback - 1` rows) as context.

In [ ]:
validation_records = []
validation_predictions = []
fitted_scalers_val = {}
fitted_models_val = {}
histories_val = {}

for symbol in symbols_to_model:
    print(f"\n--- Validation modeling for {symbol} ---")
    train_s = train_df[train_df["Symbol"] == symbol].sort_values("Date").copy().reset_index(drop=True)
    val_s = val_df[val_df["Symbol"] == symbol].sort_values("Date").copy().reset_index(drop=True)

    print("Train rows:", len(train_s), "| Validation rows:", len(val_s))

    scaler = StandardScaler()
    train_scaled = train_s.copy()
    train_scaled[feature_cols] = scaler.fit_transform(train_s[feature_cols])

    val_context = pd.concat([train_s.tail(lookback - 1), val_s], axis=0).sort_values("Date").reset_index(drop=True)
    val_context_scaled = val_context.copy()
    val_context_scaled[feature_cols] = scaler.transform(val_context[feature_cols])

    X_train, y_train, _ = make_lstm_sequences(train_scaled, feature_cols, target_col, lookback)
    X_val_all, y_val_all, meta_val_all = make_lstm_sequences(val_context_scaled, feature_cols, target_col, lookback)

    val_mask = (meta_val_all["Date"] >= val_start) & (meta_val_all["Date"] <= val_end)
    X_val = X_val_all[val_mask.values]
    y_val = y_val_all[val_mask.values]
    meta_val = meta_val_all[val_mask.values].reset_index(drop=True)

    print("X_train shape:", X_train.shape, "| X_val shape:", X_val.shape)

    naive_pred = meta_val["volatility_1D"].values
    naive_metrics = compute_metrics(y_val, naive_pred)
    validation_records.append({
        "Stage": "Validation",
        "Symbol": symbol,
        "Model": "Naive(today_vol)",
        **naive_metrics
    })

    model = build_lstm_model(
        lookback=lookback,
        n_features=len(feature_cols),
        lstm_units=lstm_units,
        learning_rate=learning_rate,
    )

    es = EarlyStopping(
        monitor="val_loss",
        patience=patience,
        restore_best_weights=True
    )

    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=[es],
        verbose=0
    )

    lstm_pred = model.predict(X_val, verbose=0).ravel()
    lstm_metrics = compute_metrics(y_val, lstm_pred)
    validation_records.append({
        "Stage": "Validation",
        "Symbol": symbol,
        "Model": "LSTM",
        **lstm_metrics
    })

    val_pred_df = meta_val[["Date", "Symbol", "volatility_1D"]].copy()
    val_pred_df["actual"] = y_val
    val_pred_df["pred_naive"] = naive_pred
    val_pred_df["pred_lstm"] = lstm_pred
    validation_predictions.append(val_pred_df)

    fitted_scalers_val[symbol] = scaler
    fitted_models_val[symbol] = model
    histories_val[symbol] = history

    print("Best validation loss epoch:", int(np.argmin(history.history["val_loss"]) + 1))

In [ ]:
validation_metrics_df = pd.DataFrame(validation_records)
validation_predictions_df = pd.concat(validation_predictions, axis=0).sort_values(["Symbol", "Date"]).reset_index(drop=True)

print("Validation metrics:")
display(validation_metrics_df.sort_values(["Symbol", "Model"]))

print("Validation predictions head:")
display(validation_predictions_df.head(5))

## Validation Plots: Actual vs Predicted

In [ ]:
fig, axes = plt.subplots(len(symbols_to_model), 1, figsize=(14, 8), sharex=False)
if len(symbols_to_model) == 1:
    axes = [axes]

for ax, symbol in zip(axes, symbols_to_model):
    p = validation_predictions_df[validation_predictions_df["Symbol"] == symbol]
    ax.plot(p["Date"], p["actual"], label="Actual", linewidth=2)
    ax.plot(p["Date"], p["pred_naive"], label="Naive", alpha=0.8)
    ax.plot(p["Date"], p["pred_lstm"], label="LSTM", alpha=0.8)
    ax.set_title(f"Validation: Actual vs Predicted ({symbol})")
    ax.set_ylabel("Next-day volatility_1D")
    ax.legend(loc="upper right")

plt.tight_layout()
plt.show()

## Validation Plots: Rolling Absolute Error

In [ ]:
rolling_window = 21

fig, axes = plt.subplots(len(symbols_to_model), 1, figsize=(14, 8), sharex=False)
if len(symbols_to_model) == 1:
    axes = [axes]

for ax, symbol in zip(axes, symbols_to_model):
    p = validation_predictions_df[validation_predictions_df["Symbol"] == symbol].copy()
    p["abs_err_naive"] = np.abs(p["actual"] - p["pred_naive"])
    p["abs_err_lstm"] = np.abs(p["actual"] - p["pred_lstm"])
    p["roll_abs_err_naive"] = p["abs_err_naive"].rolling(rolling_window, min_periods=1).mean()
    p["roll_abs_err_lstm"] = p["abs_err_lstm"].rolling(rolling_window, min_periods=1).mean()

    ax.plot(p["Date"], p["roll_abs_err_naive"], label=f"Naive {rolling_window}D rolling MAE")
    ax.plot(p["Date"], p["roll_abs_err_lstm"], label=f"LSTM {rolling_window}D rolling MAE")
    ax.set_title(f"Validation Rolling Absolute Error ({symbol})")
    ax.set_ylabel("Rolling absolute error")
    ax.legend(loc="upper right")

plt.tight_layout()
plt.show()

## Retrain on Train+Validation and Forecast Unseen

Rules applied:
- Fit new scalers on train+validation only (per stock)
- Retrain fresh LSTM models on train+validation
- Forecast unseen one day ahead sequentially
- Do not refit model daily

In [ ]:
trainval_raw = df[(df["Date"] >= data_start) & (df["Date"] <= val_end)].copy()
unseen_raw = unseen_reference_small[
    (unseen_reference_small["Date"] >= unseen_start) & (unseen_reference_small["Date"] <= unseen_end)
].copy()

trainval_features = create_engineered_features(
    trainval_raw,
    include_today=include_today,
    use_log_volatility=use_log_volatility,
)
trainval_model_df = trainval_features.dropna(subset=feature_cols + [target_col]).copy()

print("trainval_raw shape:", trainval_raw.shape)
print("unseen_raw shape  :", unseen_raw.shape)
print("trainval_model_df shape:", trainval_model_df.shape)
display(trainval_model_df.head(3))

In [ ]:
# Build unseen features with trailing history from train+validation so earliest unseen rows have valid context
history_tail_rows = max(80, lookback + 30)
unseen_features_parts = []

for symbol in symbols_to_model:
    hist_symbol = trainval_raw[trainval_raw["Symbol"] == symbol].sort_values("Date").tail(history_tail_rows)
    unseen_symbol = unseen_raw[unseen_raw["Symbol"] == symbol].sort_values("Date")

    combined_symbol = pd.concat([hist_symbol, unseen_symbol], axis=0).sort_values("Date").copy()
    combined_symbol["Symbol"] = symbol

    engineered_symbol = create_engineered_features(
        combined_symbol,
        include_today=include_today,
        use_log_volatility=use_log_volatility,
    )

    unseen_only = engineered_symbol[
        (engineered_symbol["Date"] >= unseen_start) & (engineered_symbol["Date"] <= unseen_end)
    ].copy()
    unseen_features_parts.append(unseen_only)

unseen_features = pd.concat(unseen_features_parts, axis=0).sort_values(["Symbol", "Date"]).reset_index(drop=True)
unseen_model_df = unseen_features.dropna(subset=feature_cols + [target_col]).copy()

print("unseen_features shape:", unseen_features.shape)
print("unseen_model_df shape:", unseen_model_df.shape)
display(unseen_model_df.head(3))

In [ ]:
unseen_records = []
unseen_predictions = []
fitted_scalers_unseen = {}
fitted_models_unseen = {}
histories_unseen = {}

for symbol in symbols_to_model:
    print(f"\n--- Retrain + unseen forecasting for {symbol} ---")

    trainval_s = trainval_model_df[trainval_model_df["Symbol"] == symbol].sort_values("Date").copy().reset_index(drop=True)
    unseen_s = unseen_model_df[unseen_model_df["Symbol"] == symbol].sort_values("Date").copy().reset_index(drop=True)

    print("Train+Val rows:", len(trainval_s), "| Unseen rows:", len(unseen_s))

    scaler_tv = StandardScaler()
    trainval_scaled = trainval_s.copy()
    trainval_scaled[feature_cols] = scaler_tv.fit_transform(trainval_s[feature_cols])

    X_tv, y_tv, _ = make_lstm_sequences(trainval_scaled, feature_cols, target_col, lookback)

    model_tv = build_lstm_model(
        lookback=lookback,
        n_features=len(feature_cols),
        lstm_units=lstm_units,
        learning_rate=learning_rate,
    )

    es_tv = EarlyStopping(
        monitor="loss",
        patience=patience,
        restore_best_weights=True
    )

    history_tv = model_tv.fit(
        X_tv,
        y_tv,
        epochs=epochs,
        batch_size=batch_size,
        callbacks=[es_tv],
        verbose=0
    )

    unseen_context = pd.concat([trainval_s.tail(lookback - 1), unseen_s], axis=0).sort_values("Date").reset_index(drop=True)
    unseen_context_scaled = unseen_context.copy()
    unseen_context_scaled[feature_cols] = scaler_tv.transform(unseen_context[feature_cols])

    X_unseen_all, y_unseen_all, meta_unseen_all = make_lstm_sequences(
        unseen_context_scaled,
        feature_cols,
        target_col,
        lookback
    )

    unseen_mask = (meta_unseen_all["Date"] >= unseen_start) & (meta_unseen_all["Date"] <= unseen_end)
    X_unseen = X_unseen_all[unseen_mask.values]
    y_unseen = y_unseen_all[unseen_mask.values]
    meta_unseen = meta_unseen_all[unseen_mask.values].reset_index(drop=True)

    # Sequential one-step-ahead inference without refit
    lstm_preds = []
    for i in range(len(X_unseen)):
        pred_i = model_tv.predict(X_unseen[i:i+1], verbose=0).ravel()[0]
        lstm_preds.append(pred_i)
    lstm_preds = np.asarray(lstm_preds)

    naive_preds = meta_unseen["volatility_1D"].values

    naive_metrics = compute_metrics(y_unseen, naive_preds)
    lstm_metrics = compute_metrics(y_unseen, lstm_preds)

    unseen_records.append({
        "Stage": "Unseen",
        "Symbol": symbol,
        "Model": "Naive(today_vol)",
        **naive_metrics
    })
    unseen_records.append({
        "Stage": "Unseen",
        "Symbol": symbol,
        "Model": "LSTM",
        **lstm_metrics
    })

    pred_df = meta_unseen[["Date", "Symbol", "volatility_1D"]].copy()
    pred_df["actual"] = y_unseen
    pred_df["pred_naive"] = naive_preds
    pred_df["pred_lstm"] = lstm_preds
    unseen_predictions.append(pred_df)

    fitted_scalers_unseen[symbol] = scaler_tv
    fitted_models_unseen[symbol] = model_tv
    histories_unseen[symbol] = history_tv

In [ ]:
unseen_metrics_df = pd.DataFrame(unseen_records)
unseen_predictions_df = pd.concat(unseen_predictions, axis=0).sort_values(["Symbol", "Date"]).reset_index(drop=True)

print("Unseen metrics:")
display(unseen_metrics_df.sort_values(["Symbol", "Model"]))

print("Unseen predictions head:")
display(unseen_predictions_df.head(5))

## Unseen Plots: Actual vs Predicted

In [ ]:
fig, axes = plt.subplots(len(symbols_to_model), 1, figsize=(14, 8), sharex=False)
if len(symbols_to_model) == 1:
    axes = [axes]

for ax, symbol in zip(axes, symbols_to_model):
    p = unseen_predictions_df[unseen_predictions_df["Symbol"] == symbol]
    ax.plot(p["Date"], p["actual"], label="Actual", linewidth=2)
    ax.plot(p["Date"], p["pred_naive"], label="Naive", alpha=0.8)
    ax.plot(p["Date"], p["pred_lstm"], label="LSTM", alpha=0.8)
    ax.set_title(f"Unseen: Actual vs Predicted ({symbol})")
    ax.set_ylabel("Next-day volatility_1D")
    ax.legend(loc="upper right")

plt.tight_layout()
plt.show()

## Unseen Plots: Rolling Absolute Error

In [ ]:
rolling_window = 21

fig, axes = plt.subplots(len(symbols_to_model), 1, figsize=(14, 8), sharex=False)
if len(symbols_to_model) == 1:
    axes = [axes]

for ax, symbol in zip(axes, symbols_to_model):
    p = unseen_predictions_df[unseen_predictions_df["Symbol"] == symbol].copy()
    p["abs_err_naive"] = np.abs(p["actual"] - p["pred_naive"])
    p["abs_err_lstm"] = np.abs(p["actual"] - p["pred_lstm"])
    p["roll_abs_err_naive"] = p["abs_err_naive"].rolling(rolling_window, min_periods=1).mean()
    p["roll_abs_err_lstm"] = p["abs_err_lstm"].rolling(rolling_window, min_periods=1).mean()

    ax.plot(p["Date"], p["roll_abs_err_naive"], label=f"Naive {rolling_window}D rolling MAE")
    ax.plot(p["Date"], p["roll_abs_err_lstm"], label=f"LSTM {rolling_window}D rolling MAE")
    ax.set_title(f"Unseen Rolling Absolute Error ({symbol})")
    ax.set_ylabel("Rolling absolute error")
    ax.legend(loc="upper right")

plt.tight_layout()
plt.show()

## Metric Summary Tables

In [ ]:
all_metrics_df = pd.concat([validation_metrics_df, unseen_metrics_df], axis=0).reset_index(drop=True)
all_metrics_df = all_metrics_df.sort_values(["Stage", "Symbol", "Model"]).reset_index(drop=True)

print("All metrics long-form:")
display(all_metrics_df)

print("Metrics pivot (RMSE):")
display(
    all_metrics_df.pivot_table(
        index=["Stage", "Symbol"],
        columns="Model",
        values="RMSE"
    )
)

## Final Data Checks and Samples

In [ ]:
print("Train rows by stock:")
display(train_df.groupby("Symbol").size().rename("rows").reset_index())

print("Validation rows by stock:")
display(val_df.groupby("Symbol").size().rename("rows").reset_index())

print("Unseen rows by stock (model-ready):")
display(unseen_model_df.groupby("Symbol").size().rename("rows").reset_index())

In [ ]:
print("Validation prediction sample:")
display(validation_predictions_df.groupby("Symbol").head(3))

print("Unseen prediction sample:")
display(unseen_predictions_df.groupby("Symbol").head(3))

## Notes

- Separate model trained per stock (`CTVA`, `DDOG`).
- Target is next-day `volatility_1D`.
- LSTM uses rolling sequence windows of length `lookback`.
- Feature generation, splits, and scaling follow leakage-safe rules.
- Validation is used for model assessment.
- Final unseen predictions are made sequentially without daily refits.